In [5]:
from mypackage.utilities import connect_to_db,organize_and_split_bus_line
from mypackage.mapping import reverse_combined_column_mapping
from mypackage.config import get_date_range,get_filename,bus_lines,get_specific_date,get_foldername,get_test_date
import pandas as pd
import numpy as np
import os
import shutil

# 日期参数设置
date_range=get_date_range()
cur.execute("select * from dim_org_struc")
df_org = pd.DataFrame(cur.fetchall(),columns=[desc[0] for desc in cur.description])

In [11]:
from mypackage.utilities import connect_to_db,organize_and_split_bus_line
from mypackage.mapping import reverse_combined_column_mapping
from mypackage.config import get_date_range,get_filename,bus_lines,get_specific_date,get_foldername,get_test_date
import pandas as pd
import numpy as np
import os
import shutil

# 日期参数设置
date_range=get_date_range()
# date_range=pd.date_range(start='2026-01-01', end='2026-01-31', freq='D').date
# 连接数据库获取存货数据
conn,cur= connect_to_db()
cur.execute("select * from fact_inventory where unique_lvl not like '%无归属%'")
df_inv = pd.DataFrame(cur.fetchall(),columns=[desc[0] for desc in cur.description])
df_inv=df_inv[df_inv['acct_period'].isin(date_range)]
cur.execute("select * from dim_org_struc")
df_org = pd.DataFrame(cur.fetchall(),columns=[desc[0] for desc in cur.description])
df_inv_org = pd.merge(df_inv,df_org[['unique_lvl','bus_line','short_name']],how='left',on='unique_lvl')
df_inv_matched=df_inv_org[(df_inv_org['bus_line']=='无')|(df_inv_org['bus_line']=='小POS')|(df_inv_org['bus_line']=='审核业务')]
df_inv_matched=df_inv_matched.drop(['bus_line'], axis=1)
df_inv_matched.columns = [reverse_combined_column_mapping.get(column) for column in df_inv_matched.columns]
# 将唯一层级拆分为1，2，3级
df_inv_matched=df_inv_matched.drop(['一级组织', '二级组织', '三级组织'], axis=1)
df_inv_matched[['一级组织', '二级组织', '三级组织']
            ] = df_inv_matched['唯一层级'].str.split('-', n=2, expand=True)
df_inv_matched.rename(columns={'参考金额':'金额'},inplace=True)
# 选定特定列，并组合业务线
df_inv_matched_filter=df_inv_matched[['来源编号','财报合并','财报单体','一级组织','二级组织','三级组织','物料编码','物料名称','存货类别','客户类别','客户编码','客户名称','仓库','是否为备货物料','数量(库存)','参考价(基本)','金额','6个月以内数量','6个月以内金额','6-9个月数量','6-9个月金额','9个月-1年数量','9个月-1年金额','1-2年数量','1-2年金额','2-3年数量','2-3年金额','3年以上数量','3年以上金额','会计期间','唯一层级']]
columns = bus_lines
df_inv_matched_filter.loc[:, columns] = np.nan
# 导出文件留底
df_inv_matched_filter.to_csv(r'./3-存货、应收业务拆分/存货数据拆分.csv',index=False,encoding='utf-8-sig')
conn.close()

df=df_inv_matched_filter
df_org=df_org
groups = df_inv_matched[df_inv_matched['分组简称']!="NaN"]['分组简称'].unique()
print("涉及中心:",groups)

table_name='业务线存货拆分表'
sheet_name='部门金额'
date_column='会计期间'
file_name=get_filename('业务线存货拆分表')
folder_name=get_foldername()
# folder_name="2026年01月"
# file_name="业务线存货拆分表202601.xlsx"

organize_and_split_bus_line(df, df_org, groups, date_range, date_column, table_name, sheet_name,folder_name,file_name)


涉及中心: ['smmc_center' 'smasc_center' 'pspc_center' 'psacc_center' 'dcpmc_center']
Z:\11-业务报表\8.业务线核算\智造事业群\智造管理中心\2026年02月\业务线存货拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\智造事业群\收单供应中心\2026年02月\业务线存货拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\产品中心\2026年02月\业务线存货拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\审核能力中心\2026年02月\业务线存货拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\国内渠道事业群\合伙人营销中心\2026年02月\业务线存货拆分表202602.xlsx
数据已成功写入 Excel 文件。
剩余的层级放到一个文件中


In [2]:
conn,cur= connect_to_db()
# 获取应收的数据
cur.execute("select * from fact_receivable WHERE unique_lvl not like '%无归属%'")
df_ar = pd.DataFrame(cur.fetchall(),columns=[desc[0] for desc in cur.description])
df_ar=df_ar[df_ar['acct_period'].isin(date_range)]
df_ar_org = pd.merge(df_ar,df_org[['unique_lvl','bus_line','short_name']],how='left',on='unique_lvl')
df_ar_matched=df_ar_org[(df_ar_org['bus_line']=='无')|(df_ar_org['bus_line']=='小POS')|(df_inv_org['bus_line']=='审核业务')]
# 删除内部关联方及余额为0的数据
df_ar_matched=df_ar_matched[(df_ar_matched['txn_nature']!='内部关联方往来')&(df_ar_matched['ar_balance']!=0)]

df_ar_matched=df_ar_matched.drop(['bus_line'], axis=1)
df_ar_matched.columns = [reverse_combined_column_mapping.get(column) for column in df_ar_matched.columns]
# 将唯一层级拆分为1，2，3级
df_ar_matched=df_ar_matched.drop(['一级组织', '二级组织', '三级组织'], axis=1)
df_ar_matched[['一级组织', '二级组织', '三级组织']
            ] = df_ar_matched['唯一层级'].str.split('-', n=2, expand=True)
groups = df_ar_matched[df_ar_matched['分组简称']!="NaN"]['分组简称'].unique()
print("涉及中心:",groups)
df_ar_matched=df_ar_matched.drop(['分组简称'], axis=1)
df_ar_matched=df_ar_matched.rename(columns={'应收账款余额':'金额'})
# 选择特定的列，并组合业务线
df_ar_matched = df_ar_matched[["来源编号", "财报合并", "财报单体","一级组织", "二级组织", "三级组织","一级科目", "客户编码", "往来单位", "往来性质", "客户类型", "销售部门", "销售区域",
    "赊销未核金额", "预收未核金额", "分期未核金额", "金额", "逾期金额", "未到期金额", "逾期30天以内金额", "逾期30天到90天金额",
    "逾期90天到180天金额", "逾期180天到360天金额", "逾期360天以上金额", "账龄3个月以内","账龄3-6个月","账龄6-9个月","账龄9-12个月", "账龄1-2年", "账龄2-3年",
    "账龄3年以上", "本年借方发生额", "本年贷方发生额", "销售模块", "上个月逾期金额", "逾期变动", "本年回款金额",
    "会计期间",'唯一层级', "业务大类", "业务小类","应收状态","备注"]]
columns=bus_lines
df_ar_matched.loc[:,columns] = np.nan
conn.close()
# 导出csv文件
df_ar_matched.to_csv('./3-存货、应收业务拆分/应收数据拆分.csv',index=False,encoding='utf-8-sig')

df=df_ar_matched

table_name='业务线应收拆分表'
sheet_name='部门金额'
date_column='会计期间'
file_name=get_filename('业务线应收拆分表')
folder_name=get_foldername()

organize_and_split_bus_line(df, df_org, groups, date_range, date_column, table_name, sheet_name,folder_name,file_name)

C:\Users\songsong\AppData\Local\Temp\ipykernel_37812\4201702194.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_ar_matched=df_ar_org[(df_ar_org['bus_line']=='无')|(df_ar_org['bus_line']=='小POS')|(df_inv_org['bus_line']=='审核业务')]


涉及中心: ['dcpmc_center' 'pspc_center' 'pssc_center' 'psrac_center' 'phoc_center'
 'smmc_center' 'sac_center']
Z:\11-业务报表\8.业务线核算\国内渠道事业群\合伙人营销中心\2026年02月\业务线应收拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\产品中心\2026年02月\业务线应收拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\支付安全中心\2026年02月\业务线应收拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\监管事务中心\2026年02月\业务线应收拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\国际渠道事业群\运营中心\2026年02月\业务线应收拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\智造事业群\智造管理中心\2026年02月\业务线应收拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\服务事业群\行政中心\2026年02月\业务线应收拆分表202602.xlsx
数据已成功写入 Excel 文件。
剩余的层级放到一个文件中


In [6]:


conn,cur= connect_to_db()
# 获取产品大类
cur.execute("select * from dim_material_master")
df_mat = pd.DataFrame(cur.fetchall(),columns=[desc[0] for desc in cur.description])
 
# 获取在途存货的数据
cur.execute("select * from fact_inventory_on_way WHERE unique_lvl not like '%无归属%'")
df_inv_on_way = pd.DataFrame(cur.fetchall(),columns=[desc[0] for desc in cur.description])
df_inv_on_way=df_inv_on_way[df_inv_on_way['acct_period'].isin(date_range)]
# 匹配出产品大类
df_inv_on_way = pd.merge(df_inv_on_way,df_mat[['encoding','product_major_category_report']],
                         how='left',left_on='mat_code',right_on='encoding').drop(['encoding'], axis=1)
# 根据唯一层级，匹配出业务线
df_inv_on_way_org = pd.merge(df_inv_on_way,df_org[['unique_lvl','bus_line','short_name']],how='left',on='unique_lvl')
df_inv_on_way_matched=df_inv_on_way_org[(df_inv_on_way_org['bus_line']=='无')|(df_inv_on_way_org['bus_line']=='小POS')]
# 删除内部关联方及余额为0的数据
df_inv_on_way_matched=df_inv_on_way_matched[(df_inv_on_way_matched['order_amount']!=0)]

df_inv_on_way_matched=df_inv_on_way_matched.drop(['bus_line'], axis=1)
df_inv_on_way_matched.columns = [reverse_combined_column_mapping.get(column) for column in df_inv_on_way_matched.columns]
# 将唯一层级拆分为1，2，3级
df_inv_on_way_matched=df_inv_on_way_matched.drop(['一级组织', '二级组织', '三级组织'], axis=1)
df_inv_on_way_matched[['一级组织', '二级组织', '三级组织']
            ] = df_inv_on_way_matched['唯一层级'].str.split('-', n=2, expand=True)
groups = df_inv_on_way_matched[df_inv_on_way_matched['分组简称']!="NaN"]['分组简称'].unique()
print("涉及中心:",groups)
df_inv_on_way_matched=df_inv_on_way_matched.drop(['分组简称'], axis=1)
df_inv_on_way_matched=df_inv_on_way_matched.rename(columns={'应收账款余额':'金额'})
df_inv_on_way_matched['在途订单金额']=df_inv_on_way_matched['订单金额']/df_inv_on_way_matched['订单数量']*df_inv_on_way_matched['未入库数量']
# 选择特定的列，并组合业务线
df_inv_on_way_matched = df_inv_on_way_matched[["来源编号", "财报合并", "财报单体","一级组织", "二级组织", "三级组织","订单号", "订单日期", "供应商编码", "供应商名称", "存货类别","产品大类(管报)", "物料编码", "物料名称","订单金额","在途订单金额",
    "累计付款金额", "订单数量", "累计入库数量","未入库数量", "交货日期", "会计期间","唯一层级"]]
columns=bus_lines
df_inv_on_way_matched.loc[:,columns] = np.nan
conn.close()
# 导出csv文件
df_inv_on_way_matched.to_csv('./3-存货、应收业务拆分/在途存货数据拆分.csv',index=False,encoding='utf-8-sig')

df=df_inv_on_way_matched
# 关闭数据库连接
conn.close()
table_name='业务线在途存货拆分表'
sheet_name='部门金额'
date_column='会计期间'
file_name=get_filename('业务线在途存货拆分表')
folder_name=get_foldername()
# folder_name="2026年01月"
# file_name="业务线在途存货拆分表202601.xlsx"

organize_and_split_bus_line(df, df_org, groups, date_range, date_column, table_name, sheet_name,folder_name,file_name)

涉及中心: ['smasc_center' 'smcp_center']
Z:\11-业务报表\8.业务线核算\智造事业群\收单供应中心\2026年02月\业务线在途存货拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付硬件事业群\集采中心\2026年02月\业务线在途存货拆分表202602.xlsx
数据已成功写入 Excel 文件。
剩余的层级放到一个文件中
